**Install PANNA package and requirements:**


In [ ]:
import os, sys, subprocess
from pathlib import Path
from IPython.display import Code
from ase.io import read
from ase.visualize import view
from scripts import *

main_dir = find_project_root()
print(main_dir)

In [ ]:
!{sys.executable} -m pip install {main_dir}/PANNA

**Let's visualize the Carbon dataset:**

In [ ]:
file = os.path.join(main_dir, 'dia_100/datasets/training_100_dia.xyz')
trajectory = read(file, index=":")
view(trajectory)

**Step 1: train LATTE on a small diamond dataset.**

training set: 80 atomic structures, test set: 20 structures.

In [ ]:
os.chdir(f'{main_dir}/dia_100/training')
Code(filename='train.ini')

In [ ]:
!{sys.executable} -m panna.train_jax -c train.ini

Visualize the training process:

In [ ]:
fig, axs = plt.subplots(1, 2, figsize = (12, 5))
plot_training(f'{main_dir}/dia_100/training', axs)
plot_setup(fig, axs, log=True, xlim=2000, 
            ylim=0.1, fylim=0.3,
            title='Trained on 100 Diamond')

Now, let's make prediction on Diamond test set.
Is this model able to predict more diverse dataset?

In [ ]:
os.chdir(f'{main_dir}/dia_100/test_dia')
!{sys.executable} -m panna.train_jax -c train.ini
os.chdir(f'{main_dir}/dia_100/test_div')
!{sys.executable} -m panna.train_jax -c train.ini

Test the performance of the model on familiar & unfamiliar structures:

In [ ]:
path = f'{main_dir}/dia_100/test_dia'
epoch = 50
Force_mae, Energy_mae= Force_MAE(path, epoch), Energy_MAE(path, epoch) 
print(f'At epoch {epoch}: \n Force_MAE: {Force_mae:.3f} \n Energy_MAE: {Energy_mae:.3f}')

In [ ]:
path = f'{main_dir}/dia_100/test_div'
epoch = 50
Force_mae, Energy_mae= Force_MAE(path, epoch), Energy_MAE(path, epoch) 
print(f'At epoch {epoch}: \n Force_MAE: {Force_mae:.3f} \n Energy_MAE: {Energy_mae:.3f}')

In [ ]:
dirs = [d for d in os.listdir(main_dir) if os.path.isdir(d)]
dirs = [d for d in dirs if d.startswith('di')]
colors = ['r', 'b', 'g', 'k', 'c', 'm', 'orange', 'lime']
fig, axs = plt.subplots(1, 2, figsize = (12, 5))
for d, c in zip(dirs, colors):
    plot_training(d + '/training',axs, train = False, label = d, color = c)
plot_setup(fig, axs, log=False, fylim = 0.6, xlim=2000, title='Trained on 100 Diamond')